# 02. Protocols and generics

**Tier:** Intermediate. **Why you'd use this:** You want a function that accepts "anything with `.read()`" without a shared base class, or a helper that keeps the element type through a list transform. Protocols and generics spell that for checkers.

**After this lesson you can:** explain that a `Protocol` matches structure, not inheritance; write a small generic function; use a `TypedDict` for a record shaped like a dict.
About 30 minutes. **Uses:** [Docstrings and hints](../05-functions/04-hints-lesson.ipynb), [Classes](../11-classes/01-classes-lesson.ipynb), [Tooling](01-tooling-lesson.ipynb)


## Quick review

1. Why prefer `logging` over `print` in a program you will run again?
2. Name one class of issue ruff finds that types may miss.

<details><summary>Answers</summary>

Levels, configuration, and destinations. Unused imports, style, and some bug patterns without needing full type info.
</details>


## The idea

Type hints can describe **shapes**, not only concrete classes. A `Protocol` lists the methods or attributes a value must have; any object with those members matches, even if it never subclasses the protocol. **Generics** parameterize a function or class over a type variable so `first` on a `list[str]` stays a `str`. A `TypedDict` describes the keys and value types of a dict used as a record. At runtime these are mostly documentation plus checker fuel: `TypedDict` instances are ordinary dicts.

<details><summary>Explain more</summary>

`@runtime_checkable` lets `isinstance(obj, SomeProtocol)` work for simple method checks. Without it, `isinstance` against a Protocol raises. Prefer Protocols when you care about behaviour ("has `.read()`") and base classes when you own the hierarchy.
</details>


## See it work

A generic `first` and a Protocol for anything with a `.label` property used by a formatter.


In [1]:
from typing import Protocol


def first[T](items: list[T]) -> T:
    # 1. Require a non-empty list; return the same type T.
    if not items:
        raise ValueError("empty")
    return items[0]


class Labeled(Protocol):
    @property
    def label(self) -> str: ...


class Ticket:
    def __init__(self, code: str) -> None:
        self._code = code

    @property
    def label(self) -> str:
        return self._code


def banner(item: Labeled) -> str:
    # 2. Accept any object with .label — not only Ticket.
    return f"[{item.label}]"


print(first(["a", "b"]))
print(first([10, 20]))
print(banner(Ticket("T-9")))


a
10
[T-9]


## Check yourself

Answer these from the example above before you run the cell.


1. Why is `isinstance(User("ada"), Named)` true without `User` subclassing `Named`?
2. What does `first[T]` preserve that `def first(items: list) -> object` loses for the checker?
3. What is `type` of a TypedDict value at runtime?

<details><summary>Answers</summary>

1. Structural match on `name`. 2. The element type `T`. 3. `dict`.
</details>


In [2]:
from typing import Protocol, TypedDict, runtime_checkable


@runtime_checkable
class Named(Protocol):
    name: str


class User:
    def __init__(self, name: str) -> None:
        self.name = name


class Row(TypedDict):
    id: int
    title: str


print(isinstance(User("ada"), Named))
row: Row = {"id": 1, "title": "hi"}
print(row["title"])
print(type(row).__name__)


True
hi
dict


<details><summary>Check your answers</summary>

`True` — `User` has `name`. Then `hi`. Then `dict` — TypedDict is still a dict at runtime.
</details>


## Why it works this way

Protocols are **structural** typing: the checker asks "does this value have the required members?" instead of "does it inherit this base?" That matches duck typing while still giving static tools a contract. Generics keep element types flowing through APIs so `first(nums)` stays `int` in the checker, not `Any`. Without a type parameter, helpers collapse to coarse types and callers lose autocomplete and mismatch detection. `TypedDict` models JSON-like records without forcing a class hierarchy or `__init__` boilerplate.

Cost: more annotation surface and a learning curve for Protocol versus ABC; payoff is earlier mismatch detection in editors and CI. Runtime cost is near zero—`first` is an ordinary function; values typed as a TypedDict are ordinary dicts (`type(row)` prints `dict`). `@runtime_checkable` uses a simplified `isinstance` check that looks for the listed members; it is not a full static proof.

Design reason: libraries accept many concrete types (files, sockets, mocks) that share a shape. Inheritance would force a shared base you do not own. Deep dive: pair with [Docstrings and hints](../05-functions/04-hints-lesson.ipynb) and [Classes](../11-classes/01-classes-lesson.ipynb). Evidence: `isinstance` with a runtime-checkable Protocol; `type(row)` is `dict`; generic `first` returns the first element for strings and ints; `banner` accepts `Ticket` via the Protocol shape.

A useful check when choosing Protocol versus a base class: if you control every implementer, inheritance is simple; if callers bring their own types (or tests bring fakes), Protocol keeps the door open. Generics matter most at API boundaries—helpers deep inside a module can stay unparameterized until a mismatch hurts. TypedDict shines when the wire format is already a dict and a class wrapper would only add noise.


In [3]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Closeable(Protocol):
    def close(self) -> None: ...


class Door:
    def close(self) -> None:
        return None


print(isinstance(Door(), Closeable))
print(isinstance(object(), Closeable))


True
False


## When to use it

Use Protocol for "has these methods" APIs. Use generics for containers and helpers that preserve element types. Use TypedDict for fixed-key dict records (often API JSON). Use a dataclass or class when you need methods, invariants, or identity. Don't sprinkle Protocols on every function—start from real mismatches a checker found. Don't expect TypedDict to validate keys at runtime.


## Common mistake

Treating a TypedDict as a runtime validator and expecting a bad key to raise on construction.


In [4]:
from typing import TypedDict


class Row(TypedDict):
    id: int
    title: str


bad = {"id": 1, "title": "ok", "extra": True}
print(bad["extra"])


True


TypedDict does not check keys at runtime. Validate with tests or a schema library if you need enforcement; the checker only helps when the value is typed as `Row`.


In [5]:
from typing import TypedDict


class Row(TypedDict):
    id: int
    title: str


def title_of(row: Row) -> str:
    return row["title"]


print(title_of({"id": 2, "title": "neat"}))


neat


## Try it

Add a second class with a `label` property (not a subclass of `Ticket`) and pass it to `banner`. Why does that call type-check under a Protocol but fail if `banner` required `Ticket`?


## Recap

- Protocols match structure; classes match inheritance.
- Generics keep element types through helpers.
- TypedDict describes dict records; runtime values stay dicts.

**Next:** [practice](02-protocols-and-generics-practice.ipynb)
